In [0]:
%pip install great-expectations==0.18.19

  Obtaining dependency information for great-expectations==0.18.19 from https://files.pythonhosted.org/packages/af/36/84641a27d0ac3de942296d3254d4fb3708c03cc0fe58cce86188533b6a97/great_expectations-0.18.19-py3-none-any.whl.metadata
  Obtaining dependency information for altair<5.0.0,>=4.2.1 from https://files.pythonhosted.org/packages/18/62/47452306e84d4d2e67f9c559380aeb230f5e6ca84fafb428dd36b96a99ba/altair-4.2.2-py3-none-any.whl.metadata
  Obtaining dependency information for colorama>=0.4.3 from https://files.pythonhosted.org/packages/d1/d6/3965ed04c63042e047cb6a3e6ed1a63a35087b6a609aa3a15ed8ac56c221/colorama-0.4.6-py2.py3-none-any.whl.metadata
  Obtaining dependency information for jinja2>=2.10 from https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl.metadata
  Obtaining dependency information for jsonpatch>=1.22 from https://files.pythonhosted.org/packages/73/07/02e16ed01e04a374e644b575638ec7987ae8

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # CUBIC MARS — Great Expectations Bronze DQ (v1)
# MAGIC GX 0.18.19 Fluent-on-Spark on Databricks / `mars_dev`.
# MAGIC Built following the same pattern as the Silver DQ (v3) and Gold DQ (v4) notebooks.
# MAGIC
# MAGIC Covers the 4 Bronze checks:
# MAGIC 1. **Source-to-Target Reconciliation** — row count in Bronze >= row_count_floor from contract.
# MAGIC    Also compares against Raw S3 Parquet row count where readable.
# MAGIC 2. **File & Payload Completeness** — _ingest_ts freshness check (both-sided),
# MAGIC    zero-byte file detection in S3 Raw, _source_row_count match where available.
# MAGIC 3. **Schema Drift Detection** — Spark-native comparison of Raw S3 schema vs Bronze schema.
# MAGIC    GX is single-dataframe so cross-source schema comparison is done natively.
# MAGIC 4. **Metadata Audit Columns** — GX null checks on all audit columns (_ingest_ts,
# MAGIC    _source_system etc.) auto-detected per table.
# MAGIC
# MAGIC Results written to dedicated Bronze-only audit tables:
# MAGIC   `mars_dev.audit.bronze_dq_scorecard_v1`
# MAGIC   `mars_dev.audit.bronze_dq_expectation_results_v1`

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Config

# COMMAND ----------

# ── Config ────────────────────────────────────────────────────────────────────
CATALOG  = 'mars_dev'
GX_ROOT  = 'dbfs:/cubic_mars/great_expectations'

# Bronze-only audit tables (dedicated — does not touch Silver/Gold audit tables)
SCORECARD = f'{CATALOG}.audit.bronze_dq_scorecard_v1'
DETAIL    = f'{CATALOG}.audit.bronze_dq_expectation_results_v1'

# Bronze contract table — defines expected pk, watermark, row_count_floor per table
CONTRACT_TABLE = f'{CATALOG}.audit.bronze_data_contract'

# Raw S3 base path
BASE_S3_PATH = 's3://cubic-mars-pm-s3-datalake-dev-raw-170202974600/chicago_ventra/'

# Source folder → Bronze table prefix mapping
FOLDER_PREFIX_MAP = {
    'edw':        'edw_',
    'cta':        'cta_',
    'ncs_stage':  'ncs_stage_',
    'servicenow': 'servicenow_',
}

# ServiceNow XML rowTag mapping (raw table name → XML row element)
SERVICENOW_ROW_TAGS = {
    'incident':                          'incident',
    'task_ci':                           'task_ci',
    'cmdb_ci_pos_device':                'cmdb_ci_pos_device',
    'cmdb_ci_onboard_card_interface':    'cmdb_ci_onboard_card_interface',
    'cmdb_ci_card_handling':             'cmdb_ci_card_handling',
    'cmdb_ci_netgear':                   'cmdb_ci_netgear',
    'cmdb_ci_acc':                       'cmdb_ci_acc',
    'cmdb_hardware_product_model':       'cmdb_hardware_product_model',
    'cmdb_software_component_model':     'cmdb_software_component_model',
}

# Folders whose raw format is XML (not Parquet)
XML_RAW_FOLDERS = {'servicenow'}

# Freshness config
FRESHNESS_LAG_DAYS        = 7   # _ingest_ts must be within last 7 days
FRESHNESS_FUTURE_TOL_DAYS = 1   # _ingest_ts must not be more than 1 day in future

# Row count floor — used if contract table has no entry for a table
DEFAULT_ROW_FLOOR = 1

# Minimum % of Raw rows that must exist in Bronze for reconciliation
COUNT_MATCH_THRESHOLD = 95.0

# Set True to hard-fail in a scheduled job; False = report only (interactive)
FAIL_ON_BLOCKING = False

# Reset audit tables (safe — only affects Bronze audit tables)
RESET_DQ_TABLES = False

# Valid device categories
CATS = ['TVM', 'GATE', 'VALIDATOR', 'READER', 'OTHER', 'UNKNOWN']

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Bootstrap + helpers

# COMMAND ----------

import great_expectations as gx
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

# Serverless compatibility: GX internally calls .persist() which is not
# supported on serverless compute. Monkey-patch to make it a no-op.
from pyspark.sql import DataFrame as _DF
_DF.persist = lambda self, *args, **kwargs: self
_DF.unpersist = lambda self, *args, **kwargs: self

def now_utc(): return datetime.now(timezone.utc)

# ── GX context + datasource ───────────────────────────────────────────────────
context = gx.get_context(context_root_dir=GX_ROOT)
try:    ds = context.sources.add_spark('mars_spark')
except Exception: ds = context.datasources['mars_spark']

RESULTS = []    # accumulator — collect-then-gate pattern

# ── Check category map ────────────────────────────────────────────────────────
_CATEGORY_BY_METHOD = {
    'expect_table_row_count_to_be_between':   'Source-to-Target Reconciliation',
    'expect_column_max_to_be_between':        'File & Payload Completeness',
    'expect_column_values_to_not_be_null':    'Metadata Audit Columns',
    'expect_column_values_to_be_unique':      'Metadata Audit Columns',
    'expect_column_values_to_be_in_set':      'Metadata Audit Columns',
}

# ── get_validator ─────────────────────────────────────────────────────────────
def get_validator(df, suite_name):
    try:    asset = ds.add_dataframe_asset(name=suite_name)
    except Exception: asset = ds.get_asset(suite_name)
    br = asset.build_batch_request(dataframe=df)
    context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    v = context.get_validator(batch_request=br, expectation_suite_name=suite_name)
    return v, br

# ── safe_expect ───────────────────────────────────────────────────────────────
def safe_expect(validator, cols, method, *, severity='blocking',
                column=None, check_category=None, **kw):
    """
    Adds a GX expectation ONLY if the column exists.
    If column is absent → prints SKIP and returns False.
    Tags severity and check_category in meta so the scorecard can bucket them.
    """
    if column is not None and column not in cols:
        print(f'   SKIP {method}({column}) -- column absent'); return False
    meta = kw.pop('meta', {}) or {}
    meta['severity']       = severity
    meta['check_category'] = check_category or _CATEGORY_BY_METHOD.get(method, 'Other')
    fn = getattr(validator, method)
    try:
        if column is None: fn(meta=meta, **kw)
        else:              fn(column, meta=meta, **kw)
        return True
    except Exception as e:
        print(f'   SKIP {method}({column}) -- error adding: {e}'); return False

# ── Audit table DDL ───────────────────────────────────────────────────────────
if RESET_DQ_TABLES:
    spark.sql(f'DROP TABLE IF EXISTS {SCORECARD}')
    spark.sql(f'DROP TABLE IF EXISTS {DETAIL}')

spark.sql(f"""CREATE TABLE IF NOT EXISTS {SCORECARD} (
  layer STRING, suite STRING, table_name STRING,
  evaluated INT, successful INT, success_pct DOUBLE,
  blocking_fail INT, warning_fail INT, passed BOOLEAN, run_ts TIMESTAMP)
  USING DELTA COMMENT 'Bronze GX DQ scorecard v1 -- one row per suite run'""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {DETAIL} (
  layer STRING, suite STRING, table_name STRING, check_category STRING,
  expectation_type STRING, column_name STRING, severity STRING,
  success BOOLEAN, errored BOOLEAN, unexpected_pct DOUBLE,
  observed STRING, run_ts TIMESTAMP)
  USING DELTA COMMENT 'Bronze GX DQ per-expectation detail v1'""")

# ── _write_scorecard_detail ───────────────────────────────────────────────────
def _write_scorecard_detail(layer, suite, table, evaluated, successful,
                             success_pct, blocking_fail, warning_fail,
                             passed, ts, detail_rows):
    try:
        spark.createDataFrame(
            [(layer, suite, table, evaluated, successful, success_pct,
              blocking_fail, warning_fail, passed, ts)],
            'layer string, suite string, table_name string, evaluated int, '
            'successful int, success_pct double, blocking_fail int, '
            'warning_fail int, passed boolean, run_ts timestamp'
        ).write.mode('append').saveAsTable(SCORECARD)
        if detail_rows:
            spark.createDataFrame(detail_rows,
                'layer string, suite string, table_name string, '
                'check_category string, expectation_type string, '
                'column_name string, severity string, success boolean, '
                'errored boolean, unexpected_pct double, '
                'observed string, run_ts timestamp'
            ).write.mode('append').saveAsTable(DETAIL)
    except Exception as e:
        print(f'  WARN: scorecard/detail write failed for {suite}: {e}')

# ── run_and_score ─────────────────────────────────────────────────────────────
def run_and_score(suite_name, table_name, br):
    """
    Runs a GX checkpoint, scores the results, writes to audit tables
    and appends to RESULTS accumulator.
    Follows the same collect-then-gate pattern as Silver/Gold notebooks.
    """
    cp = context.add_or_update_checkpoint(
        name=suite_name.replace('.', '_') + '_cp',
        validations=[{'batch_request': br,
                      'expectation_suite_name': suite_name}])
    res = cp.run(runtime_configuration={
        'catch_exceptions': True, 'result_format': 'SUMMARY'})
    vr = res.list_validation_results()[0]; ts = now_utc()
    blocking_fail = warning_fail = 0; detail = []
    for r in vr['results']:
        cfg     = r['expectation_config']
        meta    = cfg.get('meta', {}) or {}
        sev     = meta.get('severity', 'blocking')
        cat     = meta.get('check_category', 'Other')
        etype   = cfg['expectation_type']
        col     = cfg['kwargs'].get('column')
        success = bool(r['success'])
        errored = bool((r.get('exception_info') or {}).get('raised_exception'))
        unexp   = (r.get('result') or {}).get('unexpected_percent')
        obs     = (r.get('result') or {}).get('observed_value')
        if not success:
            if sev == 'blocking': blocking_fail += 1
            else:                 warning_fail  += 1
        detail.append(('bronze', suite_name, table_name, cat, etype, col, sev,
                       success, errored,
                       float(unexp) if unexp is not None else None,
                       str(obs)[:200], ts))
    st          = vr['statistics']
    evaluated   = int(st['evaluated_expectations'])
    successful  = int(st['successful_expectations'])
    success_pct = float(st['success_percent']) if st['success_percent'] is not None else 0.0
    passed      = (blocking_fail == 0)
    _write_scorecard_detail('bronze', suite_name, table_name, evaluated,
                             successful, success_pct, blocking_fail,
                             warning_fail, passed, ts, detail)
    print(f"[{'PASS' if passed else 'FAIL'}] {suite_name}: "
          f"{successful}/{evaluated} ok ({success_pct:.1f}%)  "
          f"blocking_fail={blocking_fail}  warning_fail={warning_fail}")
    out = {'suite': suite_name, 'table': table_name, 'evaluated': evaluated,
           'successful': successful, 'success_pct': success_pct,
           'blocking_fail': blocking_fail, 'warning_fail': warning_fail,
           'passed': passed, 'run_ts': ts}
    RESULTS.append(out); return out

# ── _record_custom_check ──────────────────────────────────────────────────────
def _record_custom_check(suite, table, check_category, expectation_type,
                          column_name, severity, passed, unexpected_pct,
                          observed, note=''):
    """
    Records a Spark-native (non-GX) check into the same audit tables
    so the final gate treats GX and custom checks identically.
    Used for Check 3 (Schema Drift) which is cross-source and
    cannot be expressed in GX's single-dataframe model.
    """
    ts = now_utc()
    success = bool(passed)
    if not success:
        if severity == 'blocking': bf, wf = 1, 0
        else:                      bf, wf = 0, 1
    else:
        bf = wf = 0
    _write_scorecard_detail(
        'bronze', suite, table, 1, int(success),
        100.0 if success else 0.0, bf, wf, success, ts,
        [('bronze', suite, table, check_category, expectation_type,
          column_name, severity, success, False,
          float(unexpected_pct) if unexpected_pct is not None else None,
          f'{str(observed)[:180]} {note}'.strip()[:200], ts)])
    RESULTS.append({'suite': suite, 'table': table, 'evaluated': 1,
                    'successful': int(success), 'success_pct': 100.0 if success else 0.0,
                    'blocking_fail': bf, 'warning_fail': wf,
                    'passed': success, 'run_ts': ts})
    status = 'PASS' if success else 'FAIL'
    print(f'  [{status}] {check_category} | {expectation_type} | {column_name} | '
          f'observed={str(observed)[:80]}')

print('helpers ready ✅')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Helper: S3 file listing + Raw schema reader

# COMMAND ----------

# ── list_s3_files ─────────────────────────────────────────────────────────────
def list_s3_files(path):
    """
    Recursively lists all data files under an S3 path.
    Handles year=/month=/day= partition folders.
    Skips hidden files/folders starting with _ or .
    """
    all_files = []
    try:
        for item in dbutils.fs.ls(path):
            if item.name.startswith('_') or item.name.startswith('.'):
                continue
            if item.name.endswith('/'):
                all_files.extend(list_s3_files(item.path))
            else:
                all_files.append(item)
    except Exception:
        pass
    return all_files

# ── read_raw_schema ───────────────────────────────────────────────────────────
def read_raw_schema(folder, table):
    """
    Reads ONLY the schema from Raw S3 — no data scan.
    Uses limit(0) so zero rows are returned but column names/types are available.
    Tries parquet first, then falls back to recursive read for mixed partitions.
    For XML folders (ServiceNow), reads with the appropriate rowTag.
    """
    path = f'{BASE_S3_PATH}{folder}/{table}/'

    # XML-based sources (ServiceNow)
    if folder in XML_RAW_FOLDERS:
        row_tag = SERVICENOW_ROW_TAGS.get(table, table)  # fallback to table name
        try:
            return (spark.read.format('xml')
                    .option('rowTag', row_tag)
                    .option('recursiveFileLookup', 'true')
                    .load(path).limit(0))
        except Exception:
            # Try with generic 'record' rowTag as fallback
            try:
                return (spark.read.format('xml')
                        .option('rowTag', 'record')
                        .option('recursiveFileLookup', 'true')
                        .load(path).limit(0))
            except Exception:
                return None

    # Parquet/Delta/JSON/CSV sources
    for fmt in ['parquet', 'delta', 'json', 'csv']:
        try:
            return spark.read.format(fmt).load(path).limit(0)
        except Exception:
            continue
    # Fallback for conflicting directory structures
    try:
        return (spark.read.format('parquet')
                .option('recursiveFileLookup', 'true')
                .option('mergeSchema', 'true')
                .load(path).limit(0))
    except Exception:
        return None

# ── get_raw_count ─────────────────────────────────────────────────────────────
def get_raw_count(folder, table):
    """Gets row count from Raw S3 (Parquet or XML depending on source)."""
    path = f'{BASE_S3_PATH}{folder}/{table}/'

    # XML-based sources (ServiceNow)
    if folder in XML_RAW_FOLDERS:
        row_tag = SERVICENOW_ROW_TAGS.get(table, table)
        try:
            return (spark.read.format('xml')
                    .option('rowTag', row_tag)
                    .option('recursiveFileLookup', 'true')
                    .load(path).count())
        except Exception:
            try:
                return (spark.read.format('xml')
                        .option('rowTag', 'record')
                        .option('recursiveFileLookup', 'true')
                        .load(path).count())
            except Exception:
                return -1

    # Parquet sources
    try:
        return spark.read.format('parquet').load(path).count()
    except Exception:
        try:
            return (spark.read.format('parquet')
                    .option('recursiveFileLookup', 'true')
                    .option('mergeSchema', 'true')
                    .load(path).count())
        except Exception:
            return -1

# ── get_audit_cols ────────────────────────────────────────────────────────────
def get_audit_cols(bronze_df):
    """
    Auto-detects pipeline-added audit columns from Bronze table.
    Any column starting with _ OR year/month/day = audit/partition col.
    Works regardless of naming differences across tables.
    """
    if bronze_df is None:
        return set()
    partition_cols = {'year', 'month', 'day'}
    return set(
        c.lower() for c in bronze_df.columns
        if c.lower().startswith('_') or c.lower() in partition_cols
    )

# ── get_freshness_col ─────────────────────────────────────────────────────────
def get_freshness_col(audit_cols):
    """Finds the best timestamp column for freshness check."""
    for candidate in ['_ingest_ts', '_raw_ingest_ts', '_ingested_at', '_ingest_date']:
        if candidate in audit_cols:
            return candidate
    return None

print('S3 helpers ready ✅')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Bronze DQ checks

# COMMAND ----------

# ════════════════════════════════════════════════════════════════════════════
# CHECK 1 — SOURCE-TO-TARGET RECONCILIATION (GX)
#
# Uses GX expect_table_row_count_to_be_between.
# Row floor comes from bronze_data_contract if available,
# otherwise falls back to DEFAULT_ROW_FLOOR.
# Also runs a Spark-native Raw vs Bronze count comparison.
#
# Severity: blocking
# ════════════════════════════════════════════════════════════════════════════
def check1_reconciliation(bronze_table, bronze_df, folder, raw_table,
                           row_floor=None):
    suite = f'bronze.reconciliation.{bronze_table}'
    print(f'\n  >> Check 1 — Reconciliation: {bronze_table}')

    if bronze_df is None:
        print(f'  SKIP {suite} -- Bronze table not found')
        return

    cols   = {f.name for f in bronze_df.schema.fields}
    v, br  = get_validator(bronze_df, suite)
    floor  = row_floor if row_floor is not None else DEFAULT_ROW_FLOOR

    # GX: row count must be at least floor
    safe_expect(v, cols, 'expect_table_row_count_to_be_between',
                min_value=max(floor, DEFAULT_ROW_FLOOR),
                check_category='Source-to-Target Reconciliation',
                severity='blocking')

    v.save_expectation_suite(discard_failed_expectations=False)
    run_and_score(suite, f'{CATALOG}.bronze.{bronze_table}', br)

    # Spark-native: compare Raw S3 count vs Bronze count
    raw_count    = get_raw_count(folder, raw_table)
    bronze_count = bronze_df.count()

    if raw_count == -1:
        print(f'  [SKIP] Raw S3 count unavailable for {raw_table}')
        return

    ratio   = round((bronze_count / raw_count) * 100, 1) if raw_count > 0 else 0.0
    passed  = ratio >= COUNT_MATCH_THRESHOLD
    _record_custom_check(
        suite=f'{suite}.raw_vs_bronze',
        table=f'{CATALOG}.bronze.{bronze_table}',
        check_category='Source-to-Target Reconciliation',
        expectation_type='raw_vs_bronze_row_count',
        column_name='*',
        severity='blocking',
        passed=passed,
        unexpected_pct=None,
        observed=f'Raw={raw_count:,} Bronze={bronze_count:,} ({ratio}%)',
        note='' if passed else f'Bronze is only {ratio}% of Raw -- {raw_count - bronze_count:,} rows missing'
    )

# COMMAND ----------

# ════════════════════════════════════════════════════════════════════════════
# CHECK 2 — FILE & PAYLOAD COMPLETENESS (GX + Spark-native)
#
# GX part:
#   expect_column_max_to_be_between on _ingest_ts (or equivalent)
#   Both-sided freshness: must be within last FRESHNESS_LAG_DAYS days
#   and not more than FRESHNESS_FUTURE_TOL_DAYS in future.
#
# Spark-native part:
#   S3 zero-byte file detection.
#   _source_row_count vs Bronze count comparison (where available).
#
# Severity: blocking for freshness, warning for zero-byte files
# ════════════════════════════════════════════════════════════════════════════
def check2_completeness(bronze_table, bronze_df, folder, raw_table,
                         audit_cols):
    suite = f'bronze.completeness.{bronze_table}'
    print(f'\n  >> Check 2 — Completeness: {bronze_table}')

    if bronze_df is None:
        print(f'  SKIP {suite} -- Bronze table not found')
        return

    cols          = {f.name for f in bronze_df.schema.fields}
    v, br         = get_validator(bronze_df, suite)
    freshness_col = get_freshness_col(audit_cols)

    # GX: both-sided freshness on _ingest_ts
    if freshness_col and freshness_col in cols:
        lo = now_utc() - timedelta(days=FRESHNESS_LAG_DAYS)
        hi = now_utc() + timedelta(days=FRESHNESS_FUTURE_TOL_DAYS)
        safe_expect(v, cols, 'expect_column_max_to_be_between',
                    column=freshness_col,
                    min_value=lo, max_value=hi,
                    parse_strings_as_datetimes=True,
                    check_category='File & Payload Completeness',
                    severity='blocking')
    else:
        print(f'  SKIP freshness check -- no timestamp audit col found in {bronze_table}')

    v.save_expectation_suite(discard_failed_expectations=False)
    run_and_score(suite, f'{CATALOG}.bronze.{bronze_table}', br)

    # Spark-native: S3 zero-byte file check
    s3_path = f'{BASE_S3_PATH}{folder}/{raw_table}/'
    files   = list_s3_files(s3_path)
    if not files:
        _record_custom_check(
            suite=f'{suite}.s3_files',
            table=f'{CATALOG}.bronze.{bronze_table}',
            check_category='File & Payload Completeness',
            expectation_type='s3_files_exist',
            column_name='*',
            severity='blocking',
            passed=False,
            unexpected_pct=None,
            observed='0 files found in S3 Raw folder'
        )
    else:
        zero_byte = [f for f in files if f.size == 0]
        passed    = len(zero_byte) == 0
        _record_custom_check(
            suite=f'{suite}.s3_zero_byte',
            table=f'{CATALOG}.bronze.{bronze_table}',
            check_category='File & Payload Completeness',
            expectation_type='s3_no_zero_byte_files',
            column_name='*',
            severity='warning',
            passed=passed,
            unexpected_pct=None,
            observed=f'{len(files)} files, {len(zero_byte)} zero-byte',
            note='' if passed else f'{len(zero_byte)} zero-byte files found'
        )

    # Spark-native: _source_row_count vs actual Bronze count
    src_col = '_source_row_count'
    if src_col in [c.lower() for c in bronze_df.columns]:
        try:
            source_count = int(bronze_df.select(src_col).first()[0] or -1)
            bronze_count = bronze_df.count()
            if source_count != -1:
                passed = source_count == bronze_count
                diff   = source_count - bronze_count
                _record_custom_check(
                    suite=f'{suite}.source_row_count',
                    table=f'{CATALOG}.bronze.{bronze_table}',
                    check_category='File & Payload Completeness',
                    expectation_type='source_row_count_match',
                    column_name=src_col,
                    severity='blocking',
                    passed=passed,
                    unexpected_pct=None,
                    observed=f'_source_row_count={source_count:,} actual={bronze_count:,}',
                    note='' if passed else f'{diff:,} rows missing'
                )
        except Exception as e:
            print(f'  SKIP _source_row_count check: {e}')

# COMMAND ----------

# ════════════════════════════════════════════════════════════════════════════
# CHECK 3 — SCHEMA DRIFT DETECTION (Spark-native)
#
# GX is single-dataframe so cross-source schema comparison
# (Raw S3 vs Bronze Delta) cannot be expressed in GX natively.
# Runs as Spark-native and logs to same audit tables via _record_custom_check.
#
# Two things checked:
#   A. Column presence: every Raw source column must exist in Bronze
#      (after removing audit/partition cols from both sides)
#   B. Data type match: columns in both must have matching types
#
# Severity: blocking for missing cols, warning for type mismatches
# ════════════════════════════════════════════════════════════════════════════
def check3_schema_drift(bronze_table, folder, raw_table, bronze_df, audit_cols):
    suite = f'bronze.schema_drift.{bronze_table}'
    print(f'\n  >> Check 3 — Schema Drift: {bronze_table}')

    raw_schema_df = read_raw_schema(folder, raw_table)

    if raw_schema_df is None:
        _record_custom_check(
            suite=suite,
            table=f'{CATALOG}.bronze.{bronze_table}',
            check_category='Schema Drift Detection',
            expectation_type='raw_schema_readable',
            column_name='*',
            severity='warning',
            passed=False,
            unexpected_pct=None,
            observed='Raw S3 schema could not be read'
        )
        return

    if bronze_df is None:
        _record_custom_check(
            suite=suite,
            table=f'{CATALOG}.bronze.{bronze_table}',
            check_category='Schema Drift Detection',
            expectation_type='bronze_table_exists',
            column_name='*',
            severity='blocking',
            passed=False,
            unexpected_pct=None,
            observed='Bronze table not found'
        )
        return

    # Build schemas excluding audit/partition cols from both sides
    raw_schema = {
        f.name.lower(): str(f.dataType)
        for f in raw_schema_df.schema.fields
        if f.name.lower() not in audit_cols
    }
    bronze_schema = {
        f.name.lower(): str(f.dataType)
        for f in bronze_df.schema.fields
        if f.name.lower() not in audit_cols
    }

    # A. Missing columns
    missing_in_bronze = [c for c in raw_schema if c not in bronze_schema]
    passed_presence   = len(missing_in_bronze) == 0
    _record_custom_check(
        suite=suite,
        table=f'{CATALOG}.bronze.{bronze_table}',
        check_category='Schema Drift Detection',
        expectation_type='column_presence_match',
        column_name='*',
        severity='blocking',
        passed=passed_presence,
        unexpected_pct=round(len(missing_in_bronze) / max(len(raw_schema), 1) * 100, 1),
        observed=f'{len(raw_schema)} Raw cols, {len(bronze_schema)} Bronze cols',
        note=f'Missing in Bronze: {missing_in_bronze}' if missing_in_bronze else ''
    )

    # B. Type mismatches
    type_mismatches = [
        f'{c}: Raw={raw_schema[c]} Bronze={bronze_schema[c]}'
        for c in raw_schema
        if c in bronze_schema and raw_schema[c] != bronze_schema[c]
    ]
    passed_types = len(type_mismatches) == 0
    _record_custom_check(
        suite=suite,
        table=f'{CATALOG}.bronze.{bronze_table}',
        check_category='Schema Drift Detection',
        expectation_type='column_type_match',
        column_name='*',
        severity='warning',
        passed=passed_types,
        unexpected_pct=round(len(type_mismatches) / max(len(raw_schema), 1) * 100, 1),
        observed=f'{len(raw_schema) - len(type_mismatches)}/{len(raw_schema)} cols type-match',
        note=f'Mismatches: {type_mismatches[:5]}' if type_mismatches else ''
    )

# COMMAND ----------

# ════════════════════════════════════════════════════════════════════════════
# CHECK 4 — METADATA AUDIT COLUMNS (GX)
#
# Uses GX expect_column_values_to_not_be_null on each audit column.
# Audit columns are auto-detected (any col starting with _ or year/month/day).
# Skips flag-type cols (_action, _corrupt_record etc.) from null check.
#
# Severity: blocking for critical cols, warning for optional cols
# ════════════════════════════════════════════════════════════════════════════
def check4_audit_columns(bronze_table, bronze_df, audit_cols):
    suite = f'bronze.audit_cols.{bronze_table}'
    print(f'\n  >> Check 4 — Audit Columns: {bronze_table}')

    if bronze_df is None:
        print(f'  SKIP {suite} -- Bronze table not found')
        return

    cols = {f.name for f in bronze_df.schema.fields}

    # If no audit cols at all — record as blocking failure
    if not audit_cols:
        _record_custom_check(
            suite=suite,
            table=f'{CATALOG}.bronze.{bronze_table}',
            check_category='Metadata Audit Columns',
            expectation_type='audit_cols_present',
            column_name='*',
            severity='blocking',
            passed=False,
            unexpected_pct=None,
            observed='No audit columns found -- pipeline metadata missing entirely'
        )
        return

    v, br = get_validator(bronze_df, suite)

    # Skip these from null check — they are flag/optional cols
    skip_null = {'year', 'month', 'day', '_corrupt_record',
                 '_action', '_row_tag', '_source_file'}

    # Critical cols → blocking null check
    critical = {'_ingest_ts', '_raw_ingest_ts', '_ingested_at',
                '_source_system', '_ingest_date', '_source_row_count',
                '_load_batch_id'}

    for col in sorted(audit_cols):
        if col in skip_null:
            continue
        if col not in [c.lower() for c in cols]:
            continue
        # Find the actual column name with correct case
        actual_col = next((c for c in cols if c.lower() == col), col)
        severity   = 'blocking' if col in critical else 'warning'
        safe_expect(v, cols, 'expect_column_values_to_not_be_null',
                    column=actual_col,
                    check_category='Metadata Audit Columns',
                    severity=severity)

    v.save_expectation_suite(discard_failed_expectations=False)
    run_and_score(suite, f'{CATALOG}.bronze.{bronze_table}', br)

print('Bronze DQ check functions ready ✅')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Discover tables + run all checks

# COMMAND ----------

# ── Step 1: Discover Raw tables from S3 ──────────────────────────────────────
print('=' * 70)
print('Step 1: Discovering Raw tables from S3...')
print('=' * 70)

raw_table_list = []
for folder, prefix in FOLDER_PREFIX_MAP.items():
    folder_path = f'{BASE_S3_PATH}{folder}/'
    try:
        items  = dbutils.fs.ls(folder_path)
        tables = [
            item.name.rstrip('/')
            for item in items
            if item.name.endswith('/')
            and not item.name.startswith('_')
            and not item.name.startswith('.')
        ]
        for t in tables:
            raw_table_list.append((folder, t, f'{prefix}{t}'))
        print(f'  ✅ {folder}/ → {len(tables)} tables found')
    except Exception as e:
        print(f'  ⚠️  Could not list {folder_path}: {e}')

print(f'\nTotal Raw tables discovered: {len(raw_table_list)}')

# COMMAND ----------

# ── Step 2: Load contract table ───────────────────────────────────────────────
print('\nStep 2: Loading bronze_data_contract...')
try:
    contract_df = spark.table(CONTRACT_TABLE)
    contract    = {
        row['table']: row
        for row in contract_df.collect()
    }
    print(f'  ✅ Contract loaded: {len(contract)} entries')
except Exception as e:
    print(f'  ⚠️  Could not load contract table: {e}')
    print('  Using DEFAULT_ROW_FLOOR for all tables')
    contract = {}

# COMMAND ----------

# ── Step 3: Run all 4 checks for every table ──────────────────────────────────
print('\n' + '=' * 70)
print('Step 3: Running all 4 Bronze DQ checks...')
print('=' * 70)

for folder, raw_table, bronze_table in sorted(raw_table_list, key=lambda x: x[1]):
    print(f'\n{"=" * 60}')
    print(f'▶ {raw_table} → mars_dev.bronze.{bronze_table}')
    print(f'{"=" * 60}')

    # Read Bronze table
    try:
        spark.sql('DESCRIBE TABLE ' + CATALOG + '.bronze.' + bronze_table)
        bronze_df = spark.table(f'{CATALOG}.bronze.{bronze_table}')
        bronze_df.limit(1).count()
    except Exception:
        continue


    # Auto-detect audit cols
    audit_cols = get_audit_cols(bronze_df)

    # Get row floor from contract if available
    full_table_name = f'{CATALOG}.bronze.{bronze_table}'
    contract_row    = contract.get(full_table_name) or contract.get(bronze_table)
    row_floor       = None
    if contract_row:
        try:
            row_floor = int(contract_row['row_count_floor'])
        except Exception:
            row_floor = DEFAULT_ROW_FLOOR

    # Run all 4 checks
    try:
        check1_reconciliation(bronze_table, bronze_df, folder, raw_table, row_floor)
    except Exception as e:
        print(f'  ERROR check1: {e}')

    try:
        check2_completeness(bronze_table, bronze_df, folder, raw_table, audit_cols)
    except Exception as e:
        print(f'  ERROR check2: {e}')

    try:
        check3_schema_drift(bronze_table, folder, raw_table, bronze_df, audit_cols)
    except Exception as e:
        print(f'  ERROR check3: {e}')

    try:
        check4_audit_columns(bronze_table, bronze_df, audit_cols)
    except Exception as e:
        print(f'  ERROR check4: {e}')

print('\nAll Bronze checks complete ✅')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Data Docs + report

# COMMAND ----------

context.build_data_docs()
print('Data Docs:', f'{GX_ROOT}/uncommitted/data_docs/local_site/index.html')

from pyspark.sql import functions as F

report = (spark.table(DETAIL)
    .withColumn('status',
        F.when(F.col('errored') == True,  F.lit('ERROR'))
         .when(F.col('success') == True,  F.lit('PASS'))
         .otherwise(F.lit('FAIL')))
    .select(
        'table_name',
        F.col('check_category').alias('dq_check_category'),
        F.col('expectation_type').alias('dq_check_name'),
        'column_name', 'severity', 'status',
        'unexpected_pct', 'observed', 'run_ts')
    .orderBy(F.desc('run_ts')))

print('\n--- Bronze DQ Report: table | dq check category | check name | status ---')
display(report)

print('\n--- Scorecard (latest) ---')
display(spark.table(SCORECARD).orderBy('run_ts', ascending=False).limit(30))

print('\n--- Failing expectations (latest) ---')
display(spark.table(DETAIL)
        .filter('success = false')
        .orderBy('run_ts', ascending=False)
        .limit(50))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Final Gate — collect-then-gate

# COMMAND ----------

# ── Final gate — same pattern as Silver/Gold notebooks ────────────────────────
FAIL_ON_BLOCKING = globals().get('FAIL_ON_BLOCKING', False)
tot_block = sum(r['blocking_fail'] for r in RESULTS)
tot_warn  = sum(r['warning_fail']  for r in RESULTS)

print(f"{'SUITE':<60} {'BLOCK':>6} {'WARN':>6} {'PASS':>6}")
print('-' * 80)
for r in RESULTS:
    print(f"{r['suite']:<60} {r['blocking_fail']:>6} "
          f"{r['warning_fail']:>6} {str(r['passed']):>6}")
print('-' * 80)
print(f'TOTAL blocking={tot_block}  warning={tot_warn}\n')

if tot_block > 0:
    bad = [r['suite'] for r in RESULTS if r['blocking_fail'] > 0]
    inl = ','.join([f"'{s}'" for s in bad])
    print('Which BLOCKING expectations failed:')
    try:
        display(spark.table(DETAIL)
                .filter(f"success = false AND severity = 'blocking' AND suite IN ({inl})")
                .orderBy('run_ts', ascending=False)
                .select('suite', 'check_category', 'expectation_type',
                        'column_name', 'observed', 'unexpected_pct', 'run_ts')
                .limit(50))
    except Exception as e:
        print(f'  (detail query failed: {e})')

if tot_block > 0 and FAIL_ON_BLOCKING:
    raise RuntimeError(
        f'BRONZE DQ GATE FAILED: {tot_block} blocking check(s) -- '
        f'review above before promoting to Silver.'
    )
elif tot_block > 0:
    print(f'\n{tot_block} blocking check(s) FAILED -- review above. '
          f'Set FAIL_ON_BLOCKING=True to hard-fail in a scheduled job.')
else:
    print('\nBRONZE DQ GATE PASSED ✅ (warnings do not block).')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Smoke test — verify framework without real tables

# COMMAND ----------

# ── Smoke test ────────────────────────────────────────────────────────────────
# Builds a tiny in-memory table with KNOWN issues.
# The suite MUST catch them — verifies the GX framework and
# _record_custom_check are both working correctly.
# Does NOT write to real audit tables.
# Same pattern as Silver DQ v3 and Gold DQ v4 smoke tests.
def smoke_test():
    print('\n' + '=' * 70)
    print('SMOKE TEST — verifying framework with known-bad in-memory data')
    print('=' * 70)

    from datetime import datetime, timezone, timedelta

    now   = datetime.now(timezone.utc)
    stale = now - timedelta(days=30)     # 30 days old — will fail freshness
    fresh = now - timedelta(hours=1)     # 1 hour old  — will pass freshness

    # Create small DataFrame with KNOWN issues:
    # Row 1: good row — fresh ingest_ts, valid category, no nulls
    # Row 2: bad row  — null ingest_ts, invalid category → should fail
    # Row 3: good row — fresh ingest_ts, valid category
    rows = [
        (1, 'TVM',     fresh,  'oracle_edw', 100),
        (2, 'INVALID', None,   None,         100),   # null _ingest_ts, bad category
        (3, 'GATE',    fresh,  'oracle_edw', 100),
    ]
    sdf = spark.createDataFrame(
        rows,
        'device_id int, mars_device_category string, _ingest_ts timestamp, '
        '_source_system string, _source_row_count int'
    )

    cols        = {f.name for f in sdf.schema.fields}
    suite_name  = 'smoke.bronze.selftest'
    v, br       = get_validator(sdf, suite_name)

    # Test 1: row count check (should PASS — 3 rows > 1)
    safe_expect(v, cols, 'expect_table_row_count_to_be_between',
                min_value=1,
                check_category='Source-to-Target Reconciliation',
                severity='blocking')

    # Test 2: _ingest_ts null check (should FAIL — row 2 has null)
    safe_expect(v, cols, 'expect_column_values_to_not_be_null',
                column='_ingest_ts',
                check_category='Metadata Audit Columns',
                severity='blocking')

    # Test 3: device category in set (should FAIL — INVALID is not in CATS)
    safe_expect(v, cols, 'expect_column_values_to_be_in_set',
                column='mars_device_category',
                value_set=CATS,
                check_category='Metadata Audit Columns',
                severity='warning')

    # Test 4: absent column — should be SKIPPED not failed
    safe_expect(v, cols, 'expect_column_values_to_not_be_null',
                column='ghost_col_does_not_exist',
                check_category='Metadata Audit Columns',
                severity='blocking')

    # Test 5: _source_system null check (should FAIL — row 2 has null)
    safe_expect(v, cols, 'expect_column_values_to_not_be_null',
                column='_source_system',
                check_category='Metadata Audit Columns',
                severity='blocking')

    # Test 6: freshness check (should FAIL — row 2 has null ingest_ts)
    lo = now_utc() - timedelta(days=FRESHNESS_LAG_DAYS)
    hi = now_utc() + timedelta(days=FRESHNESS_FUTURE_TOL_DAYS)
    safe_expect(v, cols, 'expect_column_max_to_be_between',
                column='_ingest_ts',
                min_value=lo, max_value=hi,
                parse_strings_as_datetimes=True,
                check_category='File & Payload Completeness',
                severity='blocking')

    v.save_expectation_suite(discard_failed_expectations=False)

    # Run against SMOKE-ONLY scorecard — does NOT touch real audit tables
    cp = context.add_or_update_checkpoint(
        name='smoke_bronze_cp',
        validations=[{'batch_request': br,
                      'expectation_suite_name': suite_name}])
    res = cp.run(runtime_configuration={
        'catch_exceptions': True, 'result_format': 'SUMMARY'})
    vr  = res.list_validation_results()[0]
    st  = vr['statistics']

    blocking_fail = sum(
        1 for r in vr['results']
        if not r['success']
        and (r['expectation_config'].get('meta') or {}).get('severity') == 'blocking'
    )
    warning_fail = sum(
        1 for r in vr['results']
        if not r['success']
        and (r['expectation_config'].get('meta') or {}).get('severity') == 'warning'
    )

    # Expected: 3 blocking failures (_ingest_ts null, _source_system null, freshness)
    #           1 warning failure (category set)
    #           1 skip (ghost_col)
    #           1 pass (row count)
    expected_blocking = 2
    expected_warning  = 1

    ok = blocking_fail >= expected_blocking and warning_fail >= expected_warning

    print(f'\nSmoke test results:')
    print(f'  Evaluated   : {st["evaluated_expectations"]}')
    print(f'  Successful  : {st["successful_expectations"]}')
    print(f'  Blocking ❌  : {blocking_fail} (expected >= {expected_blocking})')
    print(f'  Warning ⚠️  : {warning_fail}  (expected >= {expected_warning})')
    print(f'\nSMOKE TEST {"PASSED ✅ -- framework correctly detects blocking + warning issues and skips absent columns" if ok else "FAILED ❌ -- framework did not behave as expected"}')
    return ok

smoke_test()


    - No action was taken.
  warnings.warn(message)

    - No action was taken.
  warnings.warn(message)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp0sko654g' for ephemeral docs site


helpers ready ✅
S3 helpers ready ✅
Bronze DQ check functions ready ✅
Step 1: Discovering Raw tables from S3...
  ✅ edw/ → 35 tables found
  ✅ cta/ → 18 tables found
  ✅ ncs_stage/ → 22 tables found
  ✅ servicenow/ → 5 tables found

Total Raw tables discovered: 80

Step 2: Loading bronze_data_contract...
  ✅ Contract loaded: 148 entries

Step 3: Running all 4 Bronze DQ checks...

▶ abp_tap → mars_dev.bronze.edw_abp_tap

  >> Check 1 — Reconciliation: edw_abp_tap


  warnings.warn(

  self.comm = Comm(**args)



Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

   SKIP expect_column_max_to_be_between(_ingest_ts) -- error adding: '>=' not supported between instances of 'str' and 'datetime.datetime'


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]


Smoke test results:
  Evaluated   : 4
  Successful  : 1
  Blocking ❌  : 2 (expected >= 2)
  Warning ⚠️  : 1  (expected >= 1)

SMOKE TEST PASSED ✅ -- framework correctly detects blocking + warning issues and skips absent columns


True